In [2]:
import pandas as pd
import numpy as np
from ast import literal_eval
from tqdm.notebook import tqdm
np.random.seed(42)
n_perm = 1000

In [3]:
# Normalize across voxels
for network in ['Lang_RH']: # 'Lang', 'MD'
    for target in ['conditions', 'tasks']: 
        for train_type in ['same', 'cross']: 
            pth = f'Data/mvpa_{network}_{target}_{train_type}_pattern_details.csv'
            df = pd.read_csv(pth)
            df['Y_True'] = df['Y_True'].apply(literal_eval)
            df['Y_Pred'] = df['Y_Pred'].apply(literal_eval)
            df['Perm_Accuracies'] = pd.Series([None] * len(df), dtype=object)
            # iterate row
            for i, row in tqdm(df.iterrows(), total=len(df)):
                accuracies = []
                for perm_i in range(n_perm):
                    Y_pred_perm = np.random.permutation(row['Y_Pred'])
                    accuracy = np.mean(np.array(row['Y_True']) == Y_pred_perm)
                    accuracies.append(float(accuracy))
                df.at[i, 'Perm_Accuracies'] = accuracies
            df = df.drop(columns=['Y_True', 'Y_Pred'])
            df.to_csv(f'Data/mvpa_{network}_{target}_{train_type}_pattern_perm.csv', index=False)


  0%|          | 0/1260 [00:00<?, ?it/s]

  0%|          | 0/1260 [00:00<?, ?it/s]

  0%|          | 0/840 [00:00<?, ?it/s]

  0%|          | 0/840 [00:00<?, ?it/s]

In [4]:
perm_stats = []
for network in ['Lang_RH', 'Lang', 'MD']: 
    for target in ['conditions', 'tasks']: 
        for train_type in ['same', 'cross']: 
            pth = f'Data/mvpa_{network}_{target}_{train_type}_pattern_perm.csv'
            df = pd.read_csv(pth)
            df['Perm_Accuracies'] = df['Perm_Accuracies'].apply(literal_eval)
            for testgroup in np.unique(df['TestGroup']):
                for clf in np.unique(df['Classifier']):
                    df_i = df[(df['TestGroup'] == testgroup) & (df['Classifier'] == clf)]
                    accuracies = np.nanmean(np.array(df_i['Perm_Accuracies'].tolist()), axis=0).tolist()
                    perm_stats.append({
                        'Network': network,
                        'Target': target,
                        'TrainType': train_type,
                        'TestGroup': testgroup,
                        'Classifier': clf,
                        'Perm_Accuracies': list(accuracies)
                    })
perm_stats_df = pd.DataFrame(perm_stats)

In [5]:
perm_stats_df.to_csv(f'Data/Perm_Stats.csv', index=False)

In [36]:
from statsmodels.stats.multitest import multipletests

all_data = []

for train_type in ['same', 'cross']:
    temp_data = []
    for target in ['conditions', 'tasks']:
        for network in ['Lang', 'MD', 'Lang_RH']:
            pth = f'Data/mvpa_{network}_{target}_{train_type}_pattern.csv'
            df = pd.read_csv(pth)
            df = (
                df[['TestGroup', 'Classifier', 'Accuracy']]
                .groupby(['TestGroup', 'Classifier'])
                .mean()
                .reset_index()
            )

            df['p_uncorrected'] = df.apply(
                lambda x: np.mean(
                    np.array(
                        perm_stats_df[
                            (perm_stats_df['Network'] == network)
                            & (perm_stats_df['Target'] == target)
                            & (perm_stats_df['TrainType'] == train_type)
                            & (perm_stats_df['TestGroup'] == x['TestGroup'])
                            & (perm_stats_df['Classifier'] == x['Classifier'])
                        ]['Perm_Accuracies'].values[0]
                    )
                    >= x['Accuracy']
                ),
                axis=1,
            )

            df['target'] = target
            df['network'] = network
            df['train_type'] = train_type
            temp_data.append(df)

    combined = pd.concat(temp_data, ignore_index=True)
    all_data.append(combined)

# combine everything across train types for global correction
full_df = pd.concat(all_data, ignore_index=True)

# apply FDR correction across all p-values
full_df['p_fdr'] = multipletests(full_df['p_uncorrected'], method='fdr_bh')[1]
full_df['Sig'] = full_df['p_fdr'].apply(
    lambda x: '***' if x < 0.001 else ('**' if x < 0.01 else ('*' if x < 0.05 else 'n.s.'))
)

# Test Group as a ordered factor S, N, V1, V2, V3
testgroup_order = ['S', 'N', 'V1', 'V2', 'V3']
full_df['TestGroup'] = pd.Categorical(full_df['TestGroup'], categories=testgroup_order, ordered=True)

df_pivoted = (
    full_df.pivot(
        index=['train_type', 'target', 'network', 'TestGroup'],
        columns='Classifier',
        values=['Accuracy', 'Sig'],
    )
    .rename_axis(columns=[None, None])
    .reset_index()
)

# tidy column order
df_pivoted.columns = df_pivoted.columns.swaplevel(0, 1)
df_pivoted = df_pivoted.sort_index(axis=1, level=0, sort_remaining=False)
df_pivoted.to_csv(f'Data/Summary_aboveChance.csv', index=False)
